In [ ]:
import os
import pandas as pd
import sqlite3
import numpy as np

# Compacting the spatial current-density map

The CDDM records **667 local currents per row** (`Current_0…666`), arranged in a **29 × 23 matrix**
(S++ board between cells 5 and 6; companion Fuel Cells paper). The flow direction is the **29-axis**;
the outer ring are boundary segments the authors exclude with a correction factor (Eq. 1–2), leaving a
**27 × 21 interior**.

Goal: replace the 667 columns with a few features that keep the aging- and operating-point-relevant
structure. Two routes are compared here on steady-state maps:

1. **PCA / SVD** — data-driven modes (no assumption, but modes need interpreting).
2. **Physics features** — flow-direction gradient, heterogeneity (CoV), peak position, and lateral
   descriptors (interpretable, directly comparable to the paper).

All analysis is on each map **normalized by its own mean**, so it describes the *distribution* independent
of load level (load is just `mean_j`, kept separately).

In [ ]:
import os, sqlite3
import numpy as np
import pandas as pd
from contextlib import closing

HERE = os.getcwd()
PROJECT_DIR = os.path.dirname(HERE)
DB_path = os.path.join(PROJECT_DIR, "cathode_degradation.db")

NROW, NCOL = 29, 23                       # CDDM matrix; flow along axis 0 (29), lateral axis 1 (23)
SEG = [f"Current_{i}" for i in range(NROW * NCOL)]   # Current_0 .. Current_666

In [ ]:
# --- grid helpers -------------------------------------------------------------
def reshape_map(flat):
    """667-vector -> 29x23 grid (flow along axis 0)."""
    return np.asarray(flat, float).reshape(NROW, NCOL)

def interior(grid):
    """drop the excluded boundary ring -> 27x21."""
    return grid[1:NROW-1, 1:NCOL-1]

def boundary_corrected(flat):
    """Paper Eq. 1-2: rescale the interior so it carries the full cell current.
    (For shape/normalised analysis this cancels, but kept for correctness.)"""
    g = reshape_map(flat)
    corr = interior(g).sum() / g.sum()
    return interior(g) / corr

In [ ]:
# --- build steady-state maps: mean segment map per (age_h, load bin) ----------
def build_steady_maps(db_path=DB_path, jmin=0.25):
    sel = ",".join(f'"{c}"' for c in SEG)
    frames = []
    with closing(sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)) as con:
        ages = [r[0] for r in con.execute(
            "SELECT DISTINCT age_h FROM polarization_raw WHERE air_air_start=0 ORDER BY age_h")]
        for age in ages:                 # one age at a time keeps memory bounded
            df = pd.read_sql_query(
                f'SELECT ROUND("j.S.act [A/cm2]",1) AS j_bin, {sel} FROM polarization_raw '
                f'WHERE age_h={age} AND air_air_start=0 AND "j.S.act [A/cm2]" > {jmin}', con)
            g = df.groupby("j_bin")[SEG].mean()
            g.index = pd.MultiIndex.from_product([[age], g.index], names=["age_h", "j_bin"])
            frames.append(g)
    return pd.concat(frames)

maps = build_steady_maps()
print("steady maps:", maps.shape, "| (age_h, j_bin) index")
maps.index.to_frame(index=False).head()

In [ ]:
# interior, mean-normalised (distribution only)
I  = np.array([boundary_corrected(m).ravel() for m in maps.values])
In = I / I.mean(axis=1, keepdims=True)
print("interior maps:", In.shape, "(maps x 567 segments)")

## Route 1 — physics features

In [ ]:
def map_features(flat):
    g    = boundary_corrected(flat)
    flow = g.mean(axis=1)                 # 27-pt profile along flow
    lat  = g.mean(axis=0)                 # 21-pt profile across flow
    return pd.Series({
        "mean_j":            g.mean(),                                   # load level (kept separate)
        "heterogeneity_CoV": g.std() / g.mean(),                        # overall non-uniformity
        "flow_gradient":     np.polyfit(np.arange(len(flow)), flow/flow.mean(), 1)[0],
        "peak_pos":          int(np.argmax(flow)),                      # peak location along flow
        "lateral_gradient":  np.polyfit(np.arange(len(lat)),  lat/lat.mean(),  1)[0],
        "lateral_CoV":       lat.std() / lat.mean(),                    # cross-flow non-uniformity
    })

feat = maps.apply(lambda r: map_features(r.values), axis=1)
feat.describe().round(4)

## Route 2 — PCA on the normalised maps

In [ ]:
from sklearn.decomposition import PCA

pca = PCA().fit(In)                       # sklearn centres automatically
evr = pca.explained_variance_ratio_
print("explained variance ratio:", np.round(evr[:6], 4))
print("cumulative              :", np.round(np.cumsum(evr[:6]), 4))

for k in (1, 2, 3, 5):
    pk  = PCA(k).fit(In)
    rec = pk.inverse_transform(pk.transform(In))
    resid = np.sqrt(((In - rec) ** 2).sum() / (In ** 2).sum()) * 100
    print(f"  k={k}: reconstruction residual {resid:5.2f}%")

In [ ]:
# visualise the first 3 modes on the 27x21 interior grid
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(11, 3))
for k in range(3):
    im = ax[k].imshow(pca.components_[k].reshape(27, 21), aspect="auto", cmap="RdBu_r")
    ax[k].set_title(f"PC{k+1}  ({evr[k]*100:.0f}%)"); ax[k].set_xlabel("lateral"); ax[k].set_ylabel("flow")
    fig.colorbar(im, ax=ax[k], shrink=0.8)
plt.tight_layout(); plt.show()

## Do the physics features span the PCA subspace?

In [ ]:
scores = PCA(3).fit_transform(In)
corr = pd.DataFrame(
    {f"PC{p+1}": [abs(np.corrcoef(scores[:, p], feat[c])[0, 1]) for c in feat.columns[1:]]
     for p in range(3)},
    index=feat.columns[1:])
corr.round(2)

## Findings & recommendation

On steady-state maps spanning load and age (mean-normalised, boundary-corrected):

- The distribution is **low-rank: ~3 modes explain ~98%** of the variance (PC1 ≈ 69%, PC2 ≈ 19%, PC3 ≈ 10%).
- **PC1 ≈ the flow-direction gradient / overall heterogeneity** (|corr| ≈ 1.0 with `flow_gradient`, ≈ 0.99 with `heterogeneity_CoV`).
- **PC2 ≈ lateral (cross-flow) heterogeneity** (|corr| ≈ 0.82 with `lateral_CoV`) — this is the piece a 1-D flow profile throws away.
- **PC3 ≈ peak position** along the flow (|corr| ≈ 0.62 with `peak_pos`).

**So the two routes are nearly equivalent, and the interpretable features win.** A compact, physically
meaningful representation of the 667-segment map is:

> `mean_j` (load) · `heterogeneity_CoV` · `flow_gradient` · `peak_pos` · `lateral_CoV`

These ~5 scalars span the same subspace PCA finds (PC1–PC3) while staying directly comparable to the
paper (heterogeneity rising with age, current max shifting along the flow). Use them as the compact
columns in the modelling table; keep the PCA scores only as a cross-check that no 4th mode matters (PC4 < 2%).
Temperature, by contrast, needs only `mean` + `std` (shown separately) — no spatial decomposition.